In [1]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uproot
import pickle
import nue_booster
import importlib
importlib.reload(nue_booster)

import awkward
from typing import List, Tuple, Any, Union
from numpy.typing import NDArray

In [2]:
# All imports from data_loading.py
import hashlib
import logging
import os
import pickle
import numpy as np
import pandas as pd
import uproot
import yaml
from typing import List
import numpy as np
import awkward as ak
from typing import List, Tuple, Any, Union
from numpy.typing import NDArray
# from numu_tki import selection_1muNp 
# from numu_tki import signal_1muNp 
# from numu_tki import tki_calculators 

In [3]:
import localSettings as ls
print(ls.ntuple_path)

/exp/uboone/data/users/kpletcher/slimmedFiles


In [4]:
VARLOAD = ["selected", "nu_pdg", "backtracked_pdg","category","npi0",
           "shr_tkfit_dedx_U","shr_tkfit_dedx_V","shr_tkfit_dedx_Y",
           "shr_tkfit_nhits_U","shr_tkfit_nhits_V","shr_tkfit_nhits_Y",
           "trk_energy_tot", "shr_hits_tot", "ccnc",
           "hits_ratio", "n_tracks_contained", 
           "NeutrinoEnergy2",
           "CosmicIP","CosmicDirAll3D","CosmicIPAll3D",
           "shrmoliereavg","shrmoliererms",
           "shr_tkfit_npointsvalid","shr_tkfit_npoints", # fitted vs. all hits for shower
           "shrclusfrac0","shrclusfrac1","shrclusfrac2", # track-fitted hits / all hits
           "trkshrhitdist2", # "trkshrhitdist0","trkshrhitdist1", distance between track and shower in 2D
           "shrsubclusters0","shrsubclusters1","shrsubclusters2", # number of sub-clusters in shower
           "trk_llr_pid_score_v", # trk-PID score
           "trk_energy_proton_v", # track energy under proton hyp
           "trk_calo_energy_y_v", # track calo energy
           "reco_nu_vtx_sce_x","reco_nu_vtx_sce_y","reco_nu_vtx_sce_z",
           "nu_e", "n_showers_contained", "shr_distance", "trk_distance",
           "hits_y", "trk_len", "slnunhits", "slnhits", "shr_score", "trk_score",
           "trk_energy", "tksh_distance", "tksh_angle",
           "shr_energy_tot_cali", "evnunhits", "nslice",
           "slclustfrac", "reco_nu_vtx_x", "reco_nu_vtx_y", "reco_nu_vtx_z","contained_fraction",
           "secondshower_Y_nhit","secondshower_Y_vtxdist","secondshower_Y_dot","secondshower_Y_dir","shrclusdir2",
           "pfnhits","pfnunhits",
           "mcf_pass_ccpi0","mcf_pass_ncpi0","mcf_pass_ccnopi","mcf_pass_ncnopi","mcf_pass_cccpi","mcf_pass_nccpi",
           "flash_y_flash_matching","flash_ywidth_flash_matching","flash_z_flash_matching","flash_zwidth_flash_matching",
           "nu_centerX","nu_centerY","nu_centerZ","flash_pe_flash_matching","nu_totalCharge"
        #    "pfpplanesubclusters_U_v","pfpplanesubclusters_V_v","pfpplanesubclusters_Y_v"
          ]

# VARLOAD = ["secondshower_Y_nhit","secondshower_Y_vtxdist","secondshower_Y_dot","anglediff_Y",
#            "secondshower_V_nhit","secondshower_V_vtxdist","secondshower_V_dot","anglediff_V",
#            "secondshower_U_nhit","secondshower_U_vtxdist","secondshower_U_dot","anglediff_U"]

WEIGHTS = ["weightSpline","weightTune","weightSplineTimesTune"]

In [5]:
fold = "nuselection"
tree = "NeutrinoSelectionFilter"

# train mc samples, run4b
NU4b = 'MCC9.10_Run4b_v10_04_07_09_BNB_nu_overlay_surprise_reco2_hist.root'
NUE4b = 'MCC9.10_Run4b_v10_04_07_09_BNB_nue_overlay_surprise_reco2_hist.root'
DIRT4b = 'MCC9.10_Run4b_v10_04_07_09_BNB_dirt_surpise_reco2_hist.root'
NCPI04b = 'MCC9.10_Run4b_v10_04_07_09_BNB_NC_pi0_overlay_surprise_reco2_hist.root'

# train EXT sample, run4b
EXT4b = 'MCC9.10_Run4b_v10_04_07_09_Run4b_BNB_beam_off_surprise_reco2_hist.root'

# I don't have separate test samples, so all training samples need to be split

#ntuple_path = ls.ntuple_path
ntuple_path = "/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4b/1shower/"

u_nu4b = uproot.open(ntuple_path+NU4b)[fold][tree]
u_nue4b = uproot.open(ntuple_path+NUE4b)[fold][tree]
u_dirt4b = uproot.open(ntuple_path+DIRT4b)[fold][tree]
u_ncpi04b = uproot.open(ntuple_path+NCPI04b)[fold][tree]
u_ext4b = uproot.open(ntuple_path+EXT4b)[fold][tree]

uproot_v = [u_nu4b, u_nue4b, u_dirt4b, u_ncpi04b, u_ext4b]

nu4b = u_nu4b.arrays(VARLOAD, library="pd")
nue4b = u_nue4b.arrays(VARLOAD, library="pd")
dirt4b = u_dirt4b.arrays(VARLOAD, library="pd")
ncpi04b = u_ncpi04b.arrays(VARLOAD, library="pd")

# nu4b = u_nu4b.arrays(VARLOAD+WEIGHTS, library="pd")
# nue4b = u_nue4b.arrays(VARLOAD+WEIGHTS, library="pd")
# dirt4b = u_dirt4b.arrays(VARLOAD+WEIGHTS, library="pd")
# ncpi04b = u_ncpi04b.arrays(VARLOAD+WEIGHTS, library="pd")

ext4b = u_ext4b.arrays(VARLOAD, library="pd")
    
df_v = [nu4b, nue4b, dirt4b, ncpi04b, ext4b]

In [12]:
# print(nu4b)

In [6]:
# New block of code, gotten from data_loading.py line 570
def get_elm_from_vec_idx(
    myvec: awkward.Array, idx: Union[List[int], NDArray[Any]], fillval=np.nan
) -> NDArray[np.float64]:
    """Returns the element of a vector at position idx, where idx is a vector of indices. If idx is out of bounds, returns a filler value"""

    return np.array([pidv[tid] if ((tid < len(pidv)) & (tid >= 0)) else fillval for pidv, tid in zip(myvec, idx)])

In [10]:
# with uproot.open(data_path) as up_file:
#         up = up_file[fold][tree]
#         df = up.arrays(variables, library="pd")


def process_uproot_shower_variables(up, df):
    """Add shower variables to the dataframe using the ROOT tree."""

    df["diffY"] = df["nu_centerY"] - df["flash_y_flash_matching"]
    df["normdiffY"] = (df["nu_centerY"] - df["flash_y_flash_matching"]) / df["flash_ywidth_flash_matching"]
    df["diffZ"] = df["nu_centerZ"] - df["flash_z_flash_matching"]
    df["normdiffZ"] = (df["nu_centerZ"] - df["flash_z_flash_matching"]) / df["flash_zwidth_flash_matching"]
    df["fnl"] = 270.0*np.log((df["nu_centerX"] + df["nu_centerY"] + df["nu_centerZ"])/(df["flash_y_flash_matching"] + df["flash_z_flash_matching"])) - df["nu_centerX"]

    # this subtraction of one is needed because an ID number of 1 corresponds to the first element 
    # in an associated vector variable, but in python this is denoted by 0, hence the -1 requirement.
#     trk_id = up.arrays(["trk_id"], library="np")["trk_id"] - 1
#     shr_id = up.arrays(["shr_id"], library="np")["shr_id"] - 1

#     trk_llr_pid_v = up.arrays(["trk_llr_pid_score_v"])["trk_llr_pid_score_v"]
#     trk_calo_energy_y_v = up.arrays(["trk_calo_energy_y_v"])["trk_calo_energy_y_v"]
#     trk_energy_proton_v = up.arrays(["trk_energy_proton_v"])["trk_energy_proton_v"]

#     trk_llr_pid_v_sel = get_elm_from_vec_idx(trk_llr_pid_v, trk_id)
#     trk_calo_energy_y_sel = get_elm_from_vec_idx(trk_calo_energy_y_v, trk_id)
#     trk_energy_proton_sel = get_elm_from_vec_idx(trk_energy_proton_v, trk_id)
#     df["trkpid"] = trk_llr_pid_v_sel
#     df["trackcaloenergy"] = trk_calo_energy_y_sel
#     df["protonenergy"] = trk_energy_proton_sel

with uproot.open(ntuple_path+NU4b) as up_file:
    up=up_file[fold][tree]
    df=up.arrays(VARLOAD, library="pd")
    process_uproot_shower_variables(up,df)


# for i,df in enumerate(df_v):
#     up = uproot_v[i]

    # df["flash_y_flash_matching"] = up.arrays(["flash_y_flash_matching"])["flash_y_flash_matching"]
    # df["flash_ywidth_flash_matching"] = up.arrays(["flash_ywidth_flash_matching"])["flash_ywidth_flash_matching"]
    # df["flash_z_flash_matching"] = up.arrays(["flash_z_flash_matching"])["flash_z_flash_matching"]
    # df["flash_zwidth_flash_matching"] = up.arrays(["flash_zwidth_flash_matching"])["flash_zwidth_flash_matching"]
    # df["nu_centerX"] = up.arrays(["nu_centerX"])["nu_centerX"]
    # df["nu_centerY"] = up.arrays(["nu_centerY"])["nu_centerY"]
    # df["nu_centerZ"] = up.arrays(["nu_centerZ"])["nu_centerZ"]
    # df["flash_pe_flash_matching"] = up.arrays(["flash_pe_flash_matching"])["flash_pe_flash_matching"]
    # df["nu_totalCharge"] = up.arrays(["nu_totalCharge"])["nu_totalCharge"]

    # diffY = df["nu_centerY"] - df["flash_y_flash_matching"]

    # df["diffY"] = nu_centerY - flash_y_flash_matching

    # df["diffY"] = df["nu_centerY"] - df["flash_y_flash_matching"]
    # df["normdiffY"] = (df["nu_centerY"] - df["flash_y_flash_matching"]) / df["flash_ywidth_flash_matching"]
    # df["diffZ"] = df["nu_centerZ"] - df["flash_z_flash_matching"]
    # df["normdiffZ"] = (df["nu_centerZ"] - df["flash_z_flash_matching"]) / df["flash_zwidth_flash_matching"]
    # df["fnl"] = 270.0*np.log((df["nu_centerX"] + df["nu_centerY"] + df["nu_centerZ"])/(df["flash_y_flash_matching"] + df["flash_z_flash_matching"])) - df["nu_centerX"]

    # trk_llr_pid_v = up.arrays(["trk_llr_pid_score_v"])["trk_llr_pid_score_v"]
    # trk_energy_proton_v = up.arrays(["trk_energy_proton_v"])["trk_energy_proton_v"]
    # trk_calo_energy_y_v = up.arrays(["trk_calo_energy_y_v"])["trk_calo_energy_y_v"]
    
    # trk_id = up.arrays(["trk_id"], library="np")["trk_id"] - 1
    # shr_id = up.arrays(["shr_id"], library="np")["shr_id"] - 1

    # df["trk_llr_pid_v_sel"] = get_elm_from_vec_idx(trk_llr_pid_v, trk_id)

    # df['trkpid'] = df["trk_llr_pid_v_sel"]
    
    # trk_calo_energy_y_sel = get_elm_from_vec_idx(trk_calo_energy_y_v, trk_id)
    # trk_energy_proton_sel = get_elm_from_vec_idx(trk_energy_proton_v, trk_id)

    # df['protonenergy'] = trk_energy_proton_sel
    # df['trackcaloenergy'] = trk_calo_energy_y_sel

    # pfpplanesubclusters_U_v = up.arrays(["pfpplanesubclusters_U"])["pfpplanesubclusters_U"]
    # pfpplanesubclusters_V_v = up.arrays(["pfpplanesubclusters_V"])["pfpplanesubclusters_V"]
    # pfpplanesubclusters_Y_v = up.arrays(["pfpplanesubclusters_Y"])["pfpplanesubclusters_Y"]

    # df["shrsubclusters0"] = get_elm_from_vec_idx(pfpplanesubclusters_U_v, shr_id, 0)
    # df["shrsubclusters1"] = get_elm_from_vec_idx(pfpplanesubclusters_V_v, shr_id, 0)
    # df["shrsubclusters2"] = get_elm_from_vec_idx(pfpplanesubclusters_Y_v, shr_id, 0)

    # df['subcluster'] = df['shrsubclusters0'] + df['shrsubclusters1'] + df['shrsubclusters2']
    # df['trkfit'] = df['shr_tkfit_npointsvalid'] / df['shr_tkfit_npoints']
    # df['anglediff_Y'] = np.abs(df['secondshower_Y_dir']-df['shrclusdir2'])

TypeError: tuple indices must be integers or slices, not str

In [ ]:
# df_v_noext = [nu4b,nue4b,dirt4b,ncpi04b]

# for i,df in enumerate(df_v_noext):
#     df.loc[df["weightTune"] <= 0, "weightTune"] = 1.0
#     df.loc[df["weightTune"] == np.inf, "weightTune"] = 1.0
#     df.loc[df["weightTune"] > 100, "weightTune"] = 1.0
#     df.loc[np.isnan(df["weightTune"]) == True, "weightTune"] = 1.0
#     df.loc[df["weightSplineTimesTune"] <= 0, "weightSplineTimesTune"] = 1.0
#     df.loc[df["weightSplineTimesTune"] == np.inf, "weightSplineTimesTune"] = 1.0
#     df.loc[df["weightSplineTimesTune"] > 100, "weightSplineTimesTune"] = 1.0
#     df.loc[np.isnan(df["weightSplineTimesTune"]) == True, "weightSplineTimesTune"] = 1.0

In [ ]:
for i,df in enumerate(df_v):
    df['shr_tkfit_nhits_tot'] = (df['shr_tkfit_nhits_Y']+df['shr_tkfit_nhits_U']+df['shr_tkfit_nhits_V'])
    df['shr_tkfit_dedx_avg'] = (df['shr_tkfit_nhits_Y']*df['shr_tkfit_dedx_Y'] + df['shr_tkfit_nhits_U']*df['shr_tkfit_dedx_U'] + df['shr_tkfit_nhits_V']*df['shr_tkfit_dedx_V'])/df['shr_tkfit_nhits_tot']
    df.loc[:,'shr_tkfit_dedx_max'] = df['shr_tkfit_dedx_Y']
    df.loc[(df['shr_tkfit_nhits_U']>df['shr_tkfit_nhits_Y']),'shr_tkfit_dedx_max'] = df['shr_tkfit_dedx_U']
    df.loc[(df['shr_tkfit_nhits_V']>df['shr_tkfit_nhits_Y']) & (df['shr_tkfit_nhits_V']>df['shr_tkfit_nhits_U']),'shr_tkfit_dedx_max'] = df['shr_tkfit_dedx_V']

INTERCEPT = 0.0
SLOPE = 0.83
# define some energy-related variables
for i,df in enumerate(df_v):
    df["reco_e"] = (df["shr_energy_tot_cali"] + INTERCEPT) / SLOPE + df["trk_energy_tot"]

In [ ]:
# add back the cosmic category
for i,df in enumerate(df_v):
    df.loc[(df['category']!=1)&(df['category']!=10)&(df['category']!=11)&(df['category']!=111)&(df['slnunhits']/df['slnhits']<0.2), 'category'] = 4

In [ ]:
for i,df in enumerate(df_v):
    df["is_signal"] = 0
    df.loc[ df["category"] == 10, 'is_signal' ] = 1
    df.loc[ df["category"] == 11, 'is_signal' ] = 1

In [ ]:
MCQUERY = '((abs(nu_pdg)==12 & ccnc==0) | mcf_pass_ccpi0==1 | mcf_pass_ncpi0==1 | mcf_pass_ccnopi==1 | mcf_pass_ncnopi==1 | mcf_pass_cccpi==1 | mcf_pass_nccpi==1)'
test_mc3 = mc3.query('~'+MCQUERY)
test_mc1 = mc1.query('~'+MCQUERY)

mc3 = mc3.query(MCQUERY)
mc1 = mc1.query(MCQUERY)

train_ccpi03, test_ccpi03 = train_test_split(ccpi03, test_size=0.5, random_state=1990)
train_ext_numi, test_ext_numi = train_test_split(ext_numi, test_size=0.05, random_state=1990)
#train_ext_bnb, test_ext_bnb = train_test_split(ext, test_size=0.5, random_state=1990)

In [ ]:
train_mc = pd.concat([mc3,mc1],ignore_index=True)
train_nue = pd.concat([nuel3,nueh3,nuel1,nueh1],ignore_index=True)
train_ncpi0 = pd.concat([ncpi0s3,ncpi0s1],ignore_index=True)
train_ccpi0 = pd.concat([train_ccpi03,ccpi0s1],ignore_index=True)
train_ccnopi = pd.concat([ccnopit3,ccnopit1],ignore_index=True)
train_ncnopi = pd.concat([ncnopit3,ncnopit1],ignore_index=True)

test_mc = pd.concat([test_mc3,test_mc1],ignore_index=True)
test_nue = pd.concat([nue3,nue1],ignore_index=True)
test_ncpi0 = pd.concat([ncpi03,ncpi01],ignore_index=True)
test_ccpi0 = pd.concat([test_ccpi03,ccpi01],ignore_index=True)
test_ccnopi = pd.concat([ccnopi3,ccnopi1],ignore_index=True)
test_ncnopi = pd.concat([ncnopi3,ncnopi1],ignore_index=True)

#train_ext = pd.concat([train_ext_bnb,train_ext_numi],ignore_index=True)
#test_ext = pd.concat([test_ext_bnb,test_ext_numi],ignore_index=True)
train_ext = pd.concat([train_ext_numi,ext_numi2],ignore_index=True)
test_ext = test_ext_numi

samples = {
    "mc": (train_mc, test_mc),
    "nue": (train_nue, test_nue),
    "ncpi0": (train_ncpi0, test_ncpi0),
    "ccpi0": (train_ccpi0, test_ccpi0),
    "ccnopi": (train_ccnopi, test_ccnopi),
    "ncnopi": (train_ncnopi, test_ncnopi),
    "ext": (train_ext, test_ext),
} 

In [ ]:
for k, df in samples.items():
    df[0].loc[:,"train_weight"] = 1.
    df[1].loc[:,"train_weight"] = 1.

# override here train_weight for specific samples, if needed
samples['ccnopi'][0].loc[:,"train_weight"] = 5.
samples['ccnopi'][1].loc[:,"train_weight"] = 5.
samples['ext'][0].loc[:,"train_weight"] = 10.
samples['ext'][1].loc[:,"train_weight"] = 10.

for k, df in samples.items():
    df[0].loc[df[0]['category']==4,"train_weight"] = 10.
    df[1].loc[df[1]['category']==4,"train_weight"] = 10.

# set train_weight based on reco_e binning
reco_bins = np.linspace(0.15,1.55,15)
#reco_scaling = [8,5,3,3,2,2,2,1,1,1,1,1,1,1]
reco_scaling = [1,1,1,1,1,1,1,1,1,1,1,1,1,1]
for k, df in samples.items():
    for i, reco_bin in enumerate(reco_bins):
        if i == 0: continue
        df[0].loc[(df[0]['reco_e'] > reco_bins[i-1]) & (df[0]['reco_e'] < reco_bins[i]), 'train_weight'] = df[0]['train_weight']*reco_scaling[i-1]

In [ ]:
# variables to be trained on
TRAINVAR = ["shr_score","tksh_distance","tksh_angle",
            "shr_tkfit_dedx_max",
            "trkfit","trkpid",
            "subcluster","shrmoliereavg",
            "trkshrhitdist2","hits_ratio",
            "secondshower_Y_nhit","secondshower_Y_vtxdist","secondshower_Y_dot","anglediff_Y",
            "CosmicIPAll3D","CosmicDirAll3D",
            "is_signal","train_weight","nu_e"]

In [ ]:
print (ls.pickle_path)

In [ ]:
fig, ax = plt.subplots(1,1)

importlib.reload(nue_booster)
my_booster = nue_booster.NueBooster(samples, TRAINVAR, random_state=1990)

print (my_booster.variables)

PRESEL = "reco_e < 0.8"
PRESEL += ' and nslice == 1'
PRESEL += ' and selected == 1'
PRESEL += ' and shr_energy_tot_cali > 0.07'
PRESEL += ' and n_tracks_contained > 0'
PRESEL += ' and n_showers_contained == 1'

my_booster.set_preselection(PRESEL)

gain_imp = {}

for label, bkg_query in zip(nue_booster.labels, nue_booster.bkg_queries):
    
    preds, gain_imp[label], evals_result = my_booster.train_booster(ax, bkg_query)
    
    with open(ls.pickle_path+'booster_%s_0304_extnumi_vx_test.pickle' % label, 'wb') as booster_file:
        pickle.dump(preds, booster_file)

    #print(evals_result)
    epochs = len(evals_result['train']['error'])
    x_axis = range(0, epochs)
    # plot log loss
    fig1, ax1 = plt.subplots()
    ax1.plot(x_axis, evals_result['train']['logloss'], label='Train')
    ax1.plot(x_axis, evals_result['eval']['logloss'], label='Test')
    ax1.legend()
    plt.ylabel('Log Loss')
    plt.title('XGBoost Log Loss')
    plt.show()
    # plot classification error
    fig2, ax2 = plt.subplots()
    ax2.plot(x_axis, evals_result['train']['error'], label='Train')
    ax2.plot(x_axis, evals_result['eval']['error'], label='Test')
    ax2.legend()
    plt.ylabel('Classification Error')
    plt.title('XGBoost Classification Error')
    plt.show()
    # plot auc
    fig3, ax3 = plt.subplots()
    ax3.plot(x_axis, evals_result['train']['auc'], label='Train')
    ax3.plot(x_axis, evals_result['eval']['auc'], label='Test')
    ax3.legend()
    plt.ylabel('Area Under Curve')
    plt.title('XGBoost Area Under Curve')
    plt.show()

    
ax.set_ylim([0, 1.05])
ax.set_xlim([0, 1.0])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC')
ax.legend()
fig.tight_layout()
#fig.savefig(ls.plots_path+"roc_single.pdf")

In [ ]:
print('mc',train_mc.query(PRESEL)["train_weight"].sum())
print('nue',train_nue.query(PRESEL)["train_weight"].sum())
print('ncpi0',train_ncpi0.query(PRESEL)["train_weight"].sum())
print('ccpi0',train_ccpi0.query(PRESEL)["train_weight"].sum())
print('ccnopi',train_ccnopi.query(PRESEL)["train_weight"].sum())
print('ncnopi',train_ncnopi.query(PRESEL)["train_weight"].sum())
print('ext',train_ext.query(PRESEL)["train_weight"].sum())

In [ ]:
print('mc',test_mc.query(PRESEL)["train_weight"].sum())
print('nue',test_nue.query(PRESEL)["train_weight"].sum())
print('ncpi0',test_ncpi0.query(PRESEL)["train_weight"].sum())
print('ccpi0',test_ccpi0.query(PRESEL)["train_weight"].sum())
print('ccnopi',test_ccnopi.query(PRESEL)["train_weight"].sum())
print('ncnopi',test_ncnopi.query(PRESEL)["train_weight"].sum())
print('ext',test_ext.query(PRESEL)["train_weight"].sum())

In [ ]:
print(gain_imp['pi0'])
print(gain_imp['nonpi0'])

if 'shr_tkfit_gap10_dedx_V' in gain_imp['pi0']:
    gain_imp['pi0']['shr_tkfit_gap10_dedx_ALL'] = gain_imp['pi0']['shr_tkfit_gap10_dedx_V']+gain_imp['pi0']['shr_tkfit_gap10_dedx_U']+gain_imp['pi0']['shr_tkfit_gap10_dedx_Y']
    gain_imp['nonpi0']['shr_tkfit_gap10_dedx_ALL'] = gain_imp['nonpi0']['shr_tkfit_gap10_dedx_V']+gain_imp['nonpi0']['shr_tkfit_gap10_dedx_U']+gain_imp['nonpi0']['shr_tkfit_gap10_dedx_Y']
    del gain_imp['pi0']['shr_tkfit_gap10_dedx_V']
    del gain_imp['pi0']['shr_tkfit_gap10_dedx_U']
    del gain_imp['pi0']['shr_tkfit_gap10_dedx_Y']
    del gain_imp['nonpi0']['shr_tkfit_gap10_dedx_V']
    del gain_imp['nonpi0']['shr_tkfit_gap10_dedx_U']
    del gain_imp['nonpi0']['shr_tkfit_gap10_dedx_Y']
if 'shr_tkfit_2cm_dedx_V' in gain_imp['pi0']:
    gain_imp['pi0']['shr_tkfit_2cm_dedx_ALL'] = gain_imp['pi0']['shr_tkfit_2cm_dedx_V']+gain_imp['pi0']['shr_tkfit_2cm_dedx_U']+gain_imp['pi0']['shr_tkfit_2cm_dedx_Y']
    gain_imp['nonpi0']['shr_tkfit_2cm_dedx_ALL'] = gain_imp['nonpi0']['shr_tkfit_2cm_dedx_V']+gain_imp['nonpi0']['shr_tkfit_2cm_dedx_U']+gain_imp['nonpi0']['shr_tkfit_2cm_dedx_Y']
    del gain_imp['pi0']['shr_tkfit_2cm_dedx_V']
    del gain_imp['pi0']['shr_tkfit_2cm_dedx_U']
    del gain_imp['pi0']['shr_tkfit_2cm_dedx_Y']
    del gain_imp['nonpi0']['shr_tkfit_2cm_dedx_V']
    del gain_imp['nonpi0']['shr_tkfit_2cm_dedx_U']
    del gain_imp['nonpi0']['shr_tkfit_2cm_dedx_Y']

labels = []
pi0_imp = []
nonpi0_imp = []

for i in sorted (gain_imp['pi0'].keys()) :  
    labels.append(i)
    pi0_imp.append(gain_imp['pi0'][i])
    nonpi0_imp.append(gain_imp['nonpi0'][i])
    
x = np.arange(len(labels))  # the label locations
width = 0.35  # the width of the bars

fig, ax = plt.subplots(figsize=(8,6))
rects1 = ax.bar(x - width/2, pi0_imp, width, label='pi0')
rects2 = ax.bar(x + width/2, nonpi0_imp, width, label='nonpi0')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('total gain')
ax.set_title('BDT Variable Importance')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation = 90)
ax.legend()

plt.show()
fig.tight_layout()
#fig.savefig(ls.plots_path+"0109/bdt_var_gain.pdf")
#fig.savefig(ls.plots_path+"bdt_var_gain.pdf")

In [ ]:
d_pi0_rnk = {}
rnk = len(labels)
for w in sorted(gain_imp['pi0'], key=gain_imp['pi0'].get, reverse=True):
    d_pi0_rnk[w] = rnk
    rnk = rnk-1
print(d_pi0_rnk)

d_nonpi0_rnk = {}
rnk = len(labels)
for w in sorted(gain_imp['nonpi0'], key=gain_imp['nonpi0'].get, reverse=True):
    d_nonpi0_rnk[w] = rnk
    rnk = rnk-1
print(d_nonpi0_rnk)

pi0_rnk = []
nonpi0_rnk = []

for i in labels:  
    pi0_rnk.append(d_pi0_rnk[i])
    nonpi0_rnk.append(d_nonpi0_rnk[i])
    
x = np.arange(len(labels))  # the label locations
width = 0.35  # the width of the bars

fig, ax = plt.subplots(figsize=(8,6))
rects1 = ax.bar(x - width/2, pi0_rnk, width, label='pi0')
rects2 = ax.bar(x + width/2, nonpi0_rnk, width, label='nonpi0')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Rank (based on total gain)')
ax.set_title('BDT Variable Importance')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation = 90)
ax.legend()

plt.show()
fig.tight_layout()
#fig.savefig(ls.plots_path+"0109/bdt_var_rank.pdf")
#fig.savefig(ls.plots_path+"bdt_var_rank.pdf")